# Anytime A*: A Self-Learning Demo

## Why Anytime A*?

Anytime A* gives a quick initial path, then improves the path if more time is available.
This is useful in robotics loops where responsiveness matters.

In this notebook, you will:

1. Build a weighted grid.
2. Run weighted-A* style passes with different inflation values.
3. Compare path cost across iterations.
4. Explore how inflation controls speed/quality behavior.

## Anytime A* Essentials

- **Inflation (epsilon)** multiplies the heuristic to bias search toward speed.
- **Higher epsilon** usually means faster but less optimal paths.
- **Lower epsilon** moves toward A* quality.
- A simple anytime loop runs multiple passes while decreasing epsilon.

## 1) Helper Functions

In [ ]:
%matplotlib inline
from heapq import heappop, heappush
from typing import Dict, List, Optional, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

Point = Tuple[int, int]


def manhattan(a: Point, b: Point) -> float:
    return float(abs(a[0] - b[0]) + abs(a[1] - b[1]))


def neighbors4(cell: Point, shape: Tuple[int, int]) -> List[Point]:
    r, c = cell
    rows, cols = shape
    out: List[Point] = []
    for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols:
            out.append((nr, nc))
    return out


def weighted_a_star(cost_grid: np.ndarray, start: Point, goal: Point, epsilon: float) -> Tuple[Optional[List[Point]], int]:
    if cost_grid[start] < 0 or cost_grid[goal] < 0:
        return None, 0

    g_score: Dict[Point, float] = {start: 0.0}
    parent: Dict[Point, Optional[Point]] = {start: None}
    heap: List[Tuple[float, float, Point]] = [(epsilon * manhattan(start, goal), 0.0, start)]
    expansions = 0

    while heap:
        _, cur_cost, cur = heappop(heap)
        if cur == goal:
            break
        if cur_cost > g_score.get(cur, float('inf')):
            continue

        expansions += 1
        for nxt in neighbors4(cur, cost_grid.shape):
            cell_cost = cost_grid[nxt]
            if cell_cost < 0:
                continue
            new_cost = cur_cost + float(cell_cost)
            if new_cost < g_score.get(nxt, float('inf')):
                g_score[nxt] = new_cost
                parent[nxt] = cur
                priority = new_cost + epsilon * manhattan(nxt, goal)
                heappush(heap, (priority, new_cost, nxt))

    if goal not in parent:
        return None, expansions

    path: List[Point] = []
    cur: Optional[Point] = goal
    while cur is not None:
        path.append(cur)
        cur = parent[cur]
    path.reverse()
    return path, expansions


def path_cost(cost_grid: np.ndarray, path: Optional[List[Point]]) -> Optional[float]:
    if path is None:
        return None
    return float(sum(cost_grid[p] for p in path[1:]))


def draw_path(cost_grid: np.ndarray, start: Point, goal: Point, path: Optional[List[Point]], title: str) -> None:
    vis = np.zeros((*cost_grid.shape, 3), dtype=float)
    finite = cost_grid >= 0
    max_cost = float(cost_grid[finite].max()) if finite.any() else 1.0

    for r in range(cost_grid.shape[0]):
        for c in range(cost_grid.shape[1]):
            v = cost_grid[r, c]
            if v < 0:
                vis[r, c] = [0.0, 0.0, 0.0]
            else:
                shade = 1.0 - (v / max_cost if max_cost > 0 else 0.0) * 0.45
                vis[r, c] = [shade, shade, shade]

    if path is not None:
        for r, c in path:
            vis[r, c] = [0.2, 0.5, 0.95]

    vis[start] = [0.1, 0.8, 0.2]
    vis[goal] = [0.9, 0.2, 0.2]

    plt.figure(figsize=(6, 6))
    plt.imshow(vis, interpolation='nearest')
    plt.title(title)
    plt.xticks(range(cost_grid.shape[1]))
    plt.yticks(range(cost_grid.shape[0]))
    plt.grid(color='lightgray', linewidth=0.5)
    plt.show()

## 2) Build the Cost Map

Use `-1` for obstacles and positive values for traversal cost.

In [ ]:
cost_grid = np.array(
    [
        [1, 1, 1, -1, 2, 2, 2],
        [1, 3, 1, -1, 2, -1, 2],
        [1, 3, 1, 1, 1, -1, 2],
        [1, -1, -1, -1, 1, -1, 2],
        [1, 1, 2, 2, 1, 1, 2],
        [1, -1, 2, 3, 3, -1, 2],
        [1, 1, 1, -1, 1, 1, 1],
    ],
    dtype=float,
)

start = (0, 0)
goal = (6, 6)
eps_schedule = [2.5, 2.0, 1.5, 1.0]
print('Cost map ready.')

## 3) Run an Anytime Schedule

We run multiple weighted-A* passes while decreasing epsilon.

Observe how cost improves as epsilon approaches 1.0.

In [ ]:
results = []
for eps in eps_schedule:
    path, expansions = weighted_a_star(cost_grid, start, goal, epsilon=eps)
    cost = path_cost(cost_grid, path)
    results.append((eps, cost, expansions, path))

for eps, cost, expansions, _ in results:
    print(f'epsilon={eps:.2f} | cost={cost} | expansions={expansions}')

best_eps, best_cost, best_exp, best_path = results[-1]
draw_path(cost_grid, start, goal, best_path, title=f'Anytime A* final pass (epsilon={best_eps:.2f})')

## 4) Explore Interactively

Use the slider to test one epsilon value at a time and compare cost vs expansions.

In [ ]:
eps_slider = widgets.FloatSlider(value=1.5, min=1.0, max=3.0, step=0.25, description='epsilon')
output = widgets.Output()

def update_view(*_args):
    eps = float(eps_slider.value)
    with output:
        output.clear_output(wait=True)
        path, expansions = weighted_a_star(cost_grid, start, goal, epsilon=eps)
        cost = path_cost(cost_grid, path)
        print(f'epsilon={eps:.2f} | path found={path is not None} | cost={cost} | expansions={expansions}')
        draw_path(cost_grid, start, goal, path, title=f'Anytime A* single pass (epsilon={eps:.2f})')

eps_slider.observe(update_view, names='value')
display(widgets.VBox([eps_slider, output]))
update_view()